# AlexNet: ImageNet Classification with Deep Convolutional Neural Networks

**Aprendizaje Profundo - UNSAM**

## Implementación del Paper Seminal de Krizhevsky, Sutskever y Hinton (2012)

En este notebook implementaremos la arquitectura AlexNet que revolucionó el campo del deep learning al ganar la competencia ImageNet ILSVRC 2012 con un error top-5 del 15.3%, superando por un amplio margen al segundo lugar (26.2%).

### Contenido:
1. Introducción y contexto histórico
2. Arquitectura AlexNet original
3. Implementación en PyTorch
4. Entrenamiento en CIFAR-10 (versión adaptada)
5. Visualización de filtros y activaciones
6. Técnicas clave del paper

### Referencias:
- **Paper Original**: Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *Advances in neural information processing systems*, 25.
- **Link**: https://papers.nips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import time

# Configuración
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria GPU disponible: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Contexto Histórico y Contribuciones del Paper

### ¿Por qué AlexNet fue revolucionario?

Antes de 2012, los métodos tradicionales de computer vision dominaban ImageNet. AlexNet cambió esto demostrando que las CNNs profundas podían superar significativamente a los enfoques basados en características hechas a mano.

### Innovaciones Clave:

1. **ReLU Activation Function**
   - Primera red en usar ReLU a gran escala
   - Entrenamiento 6x más rápido que tanh
   - Evita el problema de desvanecimiento del gradiente

2. **Local Response Normalization (LRN)**
   - Normalización inspirada en neurobiología
   - Ayuda a la generalización
   - (Nota: hoy en día se usa Batch Normalization)

3. **Overlapping Pooling**
   - Pooling con stride < kernel size
   - Reduce ligeramente el error

4. **Dropout**
   - Regularización durante el entrenamiento
   - p=0.5 en las capas fully connected
   - Reduce el sobreajuste significativamente

5. **Data Augmentation**
   - Random crops y flips horizontales
   - PCA color augmentation
   - Aumenta el dataset efectivo 2048x

6. **Multi-GPU Training**
   - Entrenamiento distribuido en 2 GPUs
   - Paralelización del modelo

### Arquitectura Original:
```
Input: 224x224x3 RGB image

Conv1: 96 kernels (11x11x3), stride=4 → ReLU → MaxPool (3x3, stride=2) → LRN
Conv2: 256 kernels (5x5x48), stride=1 → ReLU → MaxPool (3x3, stride=2) → LRN
Conv3: 384 kernels (3x3x256), stride=1 → ReLU
Conv4: 384 kernels (3x3x192), stride=1 → ReLU
Conv5: 256 kernels (3x3x192), stride=1 → ReLU → MaxPool (3x3, stride=2)

FC6: 4096 neurons → ReLU → Dropout(0.5)
FC7: 4096 neurons → ReLU → Dropout(0.5)
FC8: 1000 neurons (output) → Softmax

Total: ~60 million parameters
```

## 2. Implementación de AlexNet Original

Implementaremos la arquitectura exacta del paper, con algunas adaptaciones para uso moderno.

In [ ]:
class AlexNet(nn.Module):
    """
    Implementación de AlexNet según el paper original.
    
    Arquitectura:
    - 5 capas convolucionales
    - 3 capas fully connected
    - ReLU activations
    - Local Response Normalization (LRN)
    - Dropout
    """
    
    def __init__(self, num_classes=1000):
        super(AlexNet, self).__init__()
        
        # Capas convolucionales
        self.features = nn.Sequential(
            # Conv1: (224-11)/4 + 1 = 55 → MaxPool: (55-3)/2 + 1 = 27
            nn.Conv2d(3, 96, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv2: 27 → 27 → MaxPool: (27-3)/2 + 1 = 13
            nn.Conv2d(96, 256, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv3: 13 → 13
            nn.Conv2d(256, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv4: 13 → 13
            nn.Conv2d(384, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv5: 13 → 13 → MaxPool: (13-3)/2 + 1 = 6
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        # Adaptive pooling para manejar diferentes tamaños de entrada
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        
        # Capas fully connected
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            
            nn.Dropout(p=0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            
            nn.Linear(4096, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Crear modelo
model = AlexNet(num_classes=1000)
print(model)
print(f"\nNúmero total de parámetros: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Análisis de la arquitectura
def count_parameters(model):
    """Cuenta parámetros por capa."""
    print("\n" + "="*80)
    print("ANÁLISIS DE PARÁMETROS DE ALEXNET")
    print("="*80)
    
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        total_params += params
        print(f"{name:40s} {params:>15,} parámetros")
    
    print("="*80)
    print(f"{'TOTAL':40s} {total_params:>15,} parámetros")
    print("="*80)
    
    # Calcular memoria aproximada
    param_size_mb = total_params * 4 / (1024**2)  # 4 bytes por float32
    print(f"\nMemoria aproximada del modelo: {param_size_mb:.2f} MB")
    
    return total_params

total_params = count_parameters(model)

In [ ]:
# Test forward pass con diferentes tamaños de entrada
def test_forward_pass():
    """Prueba el forward pass con diferentes tamaños de imagen."""
    model.eval()
    
    test_sizes = [(224, 224), (256, 256), (128, 128)]
    
    print("\nPrueba de forward pass con diferentes tamaños:")
    print("="*60)
    
    for size in test_sizes:
        x = torch.randn(1, 3, size[0], size[1])
        
        with torch.no_grad():
            output = model(x)
        
        print(f"Input: {x.shape} → Output: {output.shape}")
    
    print("="*60)

test_forward_pass()

## 3. Adaptación para CIFAR-10

CIFAR-10 tiene imágenes de 32x32, mucho más pequeñas que las 224x224 de ImageNet. Adaptaremos AlexNet para este dataset.

In [ ]:
class AlexNetCIFAR(nn.Module):
    """
    Versión adaptada de AlexNet para CIFAR-10 (32x32 imágenes).
    """
    
    def __init__(self, num_classes=10):
        super(AlexNetCIFAR, self).__init__()
        
        # Capas convolucionales adaptadas para imágenes más pequeñas
        self.features = nn.Sequential(
            # Conv1: kernel más pequeño y stride reducido
            nn.Conv2d(3, 64, kernel_size=5, stride=1, padding=2),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv2
            nn.Conv2d(64, 192, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5, alpha=0.0001, beta=0.75, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            
            # Conv3
            nn.Conv2d(192, 384, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv4
            nn.Conv2d(384, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            
            # Conv5
            nn.Conv2d(256, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        
        # Capas fully connected más pequeñas
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 2 * 2, 2048),
            nn.ReLU(inplace=True),
            
            nn.Dropout(p=0.5),
            nn.Linear(2048, 2048),
            nn.ReLU(inplace=True),
            
            nn.Linear(2048, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Crear modelo para CIFAR-10
model_cifar = AlexNetCIFAR(num_classes=10).to(device)
print(model_cifar)
print(f"\nNúmero total de parámetros: {sum(p.numel() for p in model_cifar.parameters()):,}")

# Test con imagen CIFAR-10
x = torch.randn(1, 3, 32, 32).to(device)
with torch.no_grad():
    output = model_cifar(x)
print(f"\nTest forward pass: Input {x.shape} → Output {output.shape}")

## 4. Preparación de Datos CIFAR-10

Implementaremos las técnicas de data augmentation mencionadas en el paper.

In [ ]:
# Data augmentation según el paper
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # Random crops
    transforms.RandomHorizontalFlip(),      # Random horizontal flips
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),  # Color jitter
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                        std=[0.2470, 0.2435, 0.2616]),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                        std=[0.2470, 0.2435, 0.2616]),
])

# Descargar y cargar CIFAR-10
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform
)

# Crear data loaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Clases de CIFAR-10
classes = ('plane', 'car', 'bird', 'cat', 'deer', 
          'dog', 'frog', 'horse', 'ship', 'truck')

print(f"Dataset de entrenamiento: {len(train_dataset)} imágenes")
print(f"Dataset de prueba: {len(test_dataset)} imágenes")
print(f"Batch size: {batch_size}")
print(f"Número de batches de entrenamiento: {len(train_loader)}")
print(f"Número de batches de prueba: {len(test_loader)}")

In [ ]:
# Visualizar algunas imágenes de entrenamiento
def imshow(img, title=None):
    """Mostrar imagen desnormalizada."""
    img = img / 2 + 0.5  # Desnormalizar
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    if title:
        plt.title(title)
    plt.axis('off')

# Obtener un batch de imágenes
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Mostrar imágenes
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for idx in range(16):
    ax = axes[idx // 8, idx % 8]
    img = images[idx]
    img = img / 2 + 0.5  # Desnormalizar
    npimg = img.numpy()
    ax.imshow(np.transpose(npimg, (1, 2, 0)))
    ax.set_title(classes[labels[idx]], fontsize=10)
    ax.axis('off')

plt.suptitle('Ejemplos de CIFAR-10 con Data Augmentation', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Entrenamiento del Modelo

Implementaremos el proceso de entrenamiento con las técnicas del paper:
- SGD con momentum (0.9)
- Learning rate decay
- Weight decay (L2 regularization)

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Entrenar una época."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Entrenamiento', leave=False)
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        pbar.set_postfix({
            'loss': f'{running_loss/total:.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    return running_loss/len(train_loader), 100.*correct/total

def validate(model, test_loader, criterion, device):
    """Validar el modelo."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(test_loader, desc='Validación', leave=False)
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    return running_loss/len(test_loader), 100.*correct/total

In [ ]:
# Configuración del entrenamiento según el paper
criterion = nn.CrossEntropyLoss()

# SGD con momentum y weight decay (como en el paper)
optimizer = optim.SGD(
    model_cifar.parameters(),
    lr=0.01,           # Learning rate inicial
    momentum=0.9,      # Momentum (paper usa 0.9)
    weight_decay=5e-4  # L2 regularization (paper usa 0.0005)
)

# Learning rate scheduler (reduce LR cuando el progreso se estanca)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=5, verbose=True
)

print("Configuración del entrenamiento:")
print(f"  Optimizador: SGD con momentum={optimizer.param_groups[0]['momentum']}")
print(f"  Learning rate inicial: {optimizer.param_groups[0]['lr']}")
print(f"  Weight decay: {optimizer.param_groups[0]['weight_decay']}")
print(f"  Criterion: CrossEntropyLoss")

In [ ]:
# Entrenamiento principal
num_epochs = 30  # Paper usa muchas más épocas, pero esto es para demostración

train_losses = []
train_accs = []
val_losses = []
val_accs = []
best_acc = 0

print(f"\nIniciando entrenamiento por {num_epochs} épocas...\n")
start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nÉpoca {epoch+1}/{num_epochs}")
    print("-" * 60)
    
    # Entrenar
    train_loss, train_acc = train_epoch(model_cifar, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # Validar
    val_loss, val_acc = validate(model_cifar, test_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Actualizar learning rate
    scheduler.step(val_loss)
    
    # Imprimir resultados
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Guardar mejor modelo
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model_cifar.state_dict(), 'alexnet_cifar10_best.pth')
        print(f"✓ Nuevo mejor modelo guardado (Acc: {best_acc:.2f}%)")

total_time = time.time() - start_time
print(f"\n\nEntrenamiento completado en {total_time/60:.2f} minutos")
print(f"Mejor precisión de validación: {best_acc:.2f}%")

In [ ]:
# Visualizar curvas de entrenamiento
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pérdida
ax1.plot(train_losses, label='Entrenamiento', linewidth=2)
ax1.plot(val_losses, label='Validación', linewidth=2)
ax1.set_xlabel('Época', fontsize=12)
ax1.set_ylabel('Pérdida', fontsize=12)
ax1.set_title('Curva de Pérdida', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Precisión
ax2.plot(train_accs, label='Entrenamiento', linewidth=2)
ax2.plot(val_accs, label='Validación', linewidth=2)
ax2.set_xlabel('Época', fontsize=12)
ax2.set_ylabel('Precisión (%)', fontsize=12)
ax2.set_title('Curva de Precisión', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('alexnet_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nResultados finales:")
print(f"  Train Accuracy: {train_accs[-1]:.2f}%")
print(f"  Val Accuracy:   {val_accs[-1]:.2f}%")
print(f"  Best Val Accuracy: {best_acc:.2f}%")

## 6. Visualización de Filtros y Activaciones

Una de las contribuciones del paper fue visualizar qué aprenden los filtros convolucionales.

In [ ]:
def visualize_first_layer_filters(model):
    """Visualizar filtros de la primera capa convolucional."""
    # Obtener los pesos de la primera capa conv
    first_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            first_conv = module
            break
    
    if first_conv is None:
        print("No se encontró capa convolucional")
        return
    
    # Obtener filtros
    filters = first_conv.weight.data.cpu().clone()
    
    # Normalizar para visualización
    filters = (filters - filters.min()) / (filters.max() - filters.min())
    
    # Visualizar primeros 64 filtros
    n_filters = min(64, filters.shape[0])
    fig, axes = plt.subplots(8, 8, figsize=(12, 12))
    
    for idx in range(n_filters):
        ax = axes[idx // 8, idx % 8]
        
        # Convertir filtro a imagen RGB
        filt = filters[idx].permute(1, 2, 0).numpy()
        ax.imshow(filt)
        ax.axis('off')
    
    plt.suptitle('Filtros de la Primera Capa Convolucional', fontsize=16)
    plt.tight_layout()
    plt.savefig('alexnet_first_layer_filters.png', dpi=150, bbox_inches='tight')
    plt.show()

# Visualizar filtros
visualize_first_layer_filters(model_cifar)

In [ ]:
def visualize_feature_maps(model, image, device):
    """Visualizar feature maps de diferentes capas."""
    model.eval()
    
    # Hook para capturar activaciones
    activations = {}
    
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output.detach()
        return hook
    
    # Registrar hooks en las capas convolucionales
    conv_layers = []
    for idx, module in enumerate(model.features.modules()):
        if isinstance(module, nn.Conv2d):
            name = f'conv{len(conv_layers)+1}'
            module.register_forward_hook(get_activation(name))
            conv_layers.append(name)
    
    # Forward pass
    with torch.no_grad():
        _ = model(image.unsqueeze(0).to(device))
    
    # Visualizar activaciones
    fig = plt.figure(figsize=(16, 12))
    
    # Imagen original
    ax = plt.subplot(3, 3, 1)
    img = image.cpu() / 2 + 0.5
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title('Imagen Original', fontsize=12)
    ax.axis('off')
    
    # Feature maps de cada capa
    for idx, layer_name in enumerate(conv_layers[:5]):
        ax = plt.subplot(3, 3, idx + 2)
        
        # Tomar promedio de los primeros 16 canales
        act = activations[layer_name][0]
        act = act[:16].mean(0).cpu()
        
        im = ax.imshow(act, cmap='viridis')
        ax.set_title(f'{layer_name} (avg 16 canales)', fontsize=11)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.suptitle('Feature Maps en Diferentes Capas', fontsize=16)
    plt.tight_layout()
    plt.savefig('alexnet_feature_maps.png', dpi=150, bbox_inches='tight')
    plt.show()

# Obtener una imagen de prueba
dataiter = iter(test_loader)
images, labels = next(dataiter)
test_image = images[0]

print(f"Clase de la imagen: {classes[labels[0]]}")
visualize_feature_maps(model_cifar, test_image, device)

## 7. Evaluación Final y Matriz de Confusión

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def evaluate_model(model, test_loader, device):
    """Evaluación completa del modelo."""
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, targets in tqdm(test_loader, desc='Evaluando'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(targets.numpy())
    
    return np.array(all_preds), np.array(all_labels)

# Cargar mejor modelo
model_cifar.load_state_dict(torch.load('alexnet_cifar10_best.pth'))

# Evaluar
predictions, labels = evaluate_model(model_cifar, test_loader, device)

# Reporte de clasificación
print("\n" + "="*80)
print("REPORTE DE CLASIFICACIÓN")
print("="*80)
print(classification_report(labels, predictions, target_names=classes))

# Matriz de confusión
cm = confusion_matrix(labels, predictions)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicción', fontsize=12)
plt.ylabel('Verdadero', fontsize=12)
plt.title('Matriz de Confusión - AlexNet en CIFAR-10', fontsize=14)
plt.tight_layout()
plt.savefig('alexnet_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Accuracy por clase
class_accuracy = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(10, 6))
plt.bar(classes, class_accuracy * 100, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Clase', fontsize=12)
plt.ylabel('Precisión (%)', fontsize=12)
plt.title('Precisión por Clase', fontsize=14)
plt.xticks(rotation=45)
plt.ylim([0, 100])
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('alexnet_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Predicciones con Ejemplos

In [ ]:
def predict_and_visualize(model, images, labels, device, num_images=12):
    """Predecir y visualizar resultados."""
    model.eval()
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    with torch.no_grad():
        outputs = model(images[:num_images].to(device))
        probs = F.softmax(outputs, dim=1)
        _, predicted = outputs.max(1)
    
    for idx in range(num_images):
        # Imagen
        img = images[idx].cpu() / 2 + 0.5
        axes[idx].imshow(img.permute(1, 2, 0))
        
        # Título con predicción
        true_label = classes[labels[idx]]
        pred_label = classes[predicted[idx]]
        confidence = probs[idx, predicted[idx]].item() * 100
        
        color = 'green' if predicted[idx] == labels[idx] else 'red'
        title = f"Real: {true_label}\nPred: {pred_label} ({confidence:.1f}%)"
        axes[idx].set_title(title, fontsize=10, color=color, weight='bold')
        axes[idx].axis('off')
    
    plt.suptitle('Predicciones de AlexNet (Verde=Correcto, Rojo=Error)', 
                fontsize=16, y=0.98)
    plt.tight_layout()
    plt.savefig('alexnet_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

# Obtener batch de imágenes
dataiter = iter(test_loader)
images, labels = next(dataiter)

predict_and_visualize(model_cifar, images, labels, device)

## 9. Resumen y Conclusiones

### Logros de AlexNet:

1. **Rendimiento sin precedentes**: Redujo el error top-5 de 26.2% a 15.3% en ImageNet
2. **Popularizó las GPUs**: Demostró que las GPUs eran esenciales para deep learning
3. **Técnicas innovadoras**: ReLU, Dropout, Data Augmentation se volvieron estándar
4. **Inspiró investigación**: Inició la era moderna del deep learning

### Diferencias entre el Paper Original y Nuestra Implementación:

| Aspecto | Paper Original | Nuestra Implementación |
|---------|----------------|------------------------|
| Dataset | ImageNet (1.2M imágenes, 1000 clases) | CIFAR-10 (60K imágenes, 10 clases) |
| Tamaño de entrada | 224×224 | 32×32 |
| Parámetros | ~60M | ~10M |
| GPUs | 2× GTX 580 (3GB cada una) | 1× GPU moderna |
| Tiempo de entrenamiento | ~6 días | ~1 hora |
| Épocas | 90 | 30 |

### Lecciones Clave:

1. **Profundidad importa**: Las redes profundas capturan jerarquías de características
2. **Regularización es crucial**: Dropout y data augmentation previenen overfitting
3. **Activación ReLU**: Más rápida y efectiva que sigmoides/tanh
4. **Data augmentation**: Multiplica efectivamente el tamaño del dataset
5. **GPU computing**: Habilitó el entrenamiento de redes grandes

### Evolución Post-AlexNet:

- **VGGNet (2014)**: Redes más profundas y uniformes
- **GoogLeNet/Inception (2014)**: Módulos inception, arquitecturas eficientes
- **ResNet (2015)**: Skip connections, redes ultra-profundas (152+ capas)
- **EfficientNet (2019)**: Scaling balanceado de ancho, profundidad y resolución

### Impacto en Computer Vision:

AlexNet no solo ganó la competencia, sino que cambió fundamentalmente cómo se aborda computer vision:
- De características hechas a mano a aprendizaje de características
- De modelos superficiales a arquitecturas profundas
- De CPUs a GPUs como herramienta estándar
- De datasets pequeños a datasets masivos

## Ejercicios

1. **Experimenta con la arquitectura**:
   - Añade Batch Normalization en lugar de LRN
   - Prueba diferentes números de filtros
   - Cambia las funciones de activación

2. **Data Augmentation avanzado**:
   - Implementa PCA color augmentation del paper
   - Prueba técnicas modernas como Cutout, Mixup, CutMix

3. **Transfer Learning**:
   - Carga un AlexNet pre-entrenado en ImageNet
   - Fine-tunea en CIFAR-10
   - Compara con entrenar desde cero

4. **Visualización**:
   - Implementa Grad-CAM para visualizar qué mira la red
   - Visualiza filtros de capas más profundas

5. **Optimización**:
   - Prueba Adam optimizer en lugar de SGD
   - Implementa learning rate warmup
   - Experimenta con diferentes schedules de LR

In [ ]:
# Espacio para experimentación